In [ ]:
import duckdb
import os

os.chdir('..')

print("Working directory:", os.getcwd())
# Create an in-memory DuckDB connection
con = duckdb.connect()

print("DuckDB version:", duckdb.__version__)

Working directory: c:\Users\Tobi\Desktop\CSHW\TransitKPIFramework
DuckDB version: 1.4.4


How many routes exist and what types are they?

In [2]:
con.sql("""
    SELECT 
        route_type,
        COUNT(*) as num_routes
    FROM read_csv_auto('data/google_bus/routes.txt')
    GROUP BY route_type
    ORDER BY num_routes DESC
""").df()

,route_type,num_routes
0,3,156
1,0,8
2,1,5
3,11,3


How many trips per route? (top 20)

In [3]:
con.sql("""
    SELECT 
        r.route_short_name,
        r.route_long_name,
        COUNT(t.trip_id) as num_trips
    FROM read_csv_auto('data/google_bus/routes.txt') r
    JOIN read_csv_auto('data/google_bus/trips.txt') t
        ON r.route_id = t.route_id
    GROUP BY r.route_short_name, r.route_long_name
    ORDER BY num_trips DESC
    LIMIT 20
""").df()

,route_short_name,route_long_name,num_trips
0,L1,Market-Frankford Line All Stops,746
1,B1,Broad Street Line Local,704
2,T Bus,City Hall to 40th-Market,648
3,T5,13th St to 80th St/Eastwick,632
4,T3,13th St to Yeadon or Darby TC,614
5,66,Frankford TC to Frankford-Knights,610
6,52,49th-Woodlnd to 54th-City/Parksd Lp,599
7,T1,13th St to 63rd-Malvern/Overbrook,578
8,T2,13th St to 61st-Baltimore/Angora,554
9,G1,63rd-Girard to Richmond-Westmorelnd,536


What does a single trip's schedule look like?

In [4]:
con.sql("""
    SELECT 
        trip_id,
        stop_sequence,
        arrival_time,
        departure_time,
        stop_id
    FROM read_csv_auto('data/google_bus/stop_times.txt')
    WHERE trip_id = (
        SELECT trip_id 
        FROM read_csv_auto('data/google_bus/stop_times.txt') 
        LIMIT 1
    )
    ORDER BY stop_sequence
""").df()

,trip_id,stop_sequence,arrival_time,departure_time,stop_id
0,656953,1,06:05:00,06:05:00,809
1,656953,2,06:05:36,06:05:36,27684
2,656953,3,06:07:20,06:07:20,2321
3,656953,4,06:08:18,06:08:18,16404
4,656953,5,06:09:26,06:09:26,16405
5,656953,6,06:09:42,06:09:42,31493
6,656953,7,06:09:54,06:09:54,21263
7,656953,8,06:10:41,06:10:41,16406
8,656953,9,06:11:30,06:11:30,29048
9,656953,10,06:12:00,06:12:00,19544


In [5]:
con.sql("""
    SELECT
        r.route_short_name,
        r.route_long_name,
        r.route_type,
        t.trip_id,
        t.service_id,
        t.direction_id
    FROM read_csv_auto('data/google_bus/trips.txt') t
    JOIN read_csv_auto('data/google_bus/routes.txt') r
        ON t.route_id = r.route_id
    WHERE t.trip_id = 656953
""").df()

,route_short_name,route_long_name,route_type,trip_id,service_id,direction_id
0,310,Horsham Loop,3,656953,22,0


In [6]:
con.sql("""
    SELECT 
        r.route_short_name,
        r.route_long_name,
        COUNT(t.trip_id) as num_trips
    FROM read_csv_auto('data/google_bus/routes.txt') r
    JOIN read_csv_auto('data/google_bus/trips.txt') t
        ON r.route_id = t.route_id
    WHERE r.route_type = 3
    GROUP BY r.route_short_name, r.route_long_name
    ORDER BY num_trips DESC
    LIMIT 20
""").df()

,route_short_name,route_long_name,num_trips
0,T Bus,City Hall to 40th-Market,648
1,52,49th-Woodlnd to 54th-City/Parksd Lp,599
2,23,11th Market to Chestnut Hill,523
3,33,5th-Market to 23rd-Venango,505
4,42,2nd-Spruce to 61st-Pine/Wycombe,504
5,47,Whitman Plaza to 5th-Godfrey,499
6,45,Broad-Oregon to 12th-Vine,478
7,17,Front-Mkt to 20-Johnston/Broad-Pat,467
8,63,Lanknau/Overbrk to ColCom/FdCtr,466
9,18,Fox Chase to Cedarbrook Plaza,465


In [7]:
con.sql("""
    SELECT
        r.route_short_name,
        r.route_long_name,
        COUNT(t.trip_id) as weekday_trips
    FROM read_csv_auto('data/google_bus/routes.txt') r
    JOIN read_csv_auto('data/google_bus/trips.txt') t
        ON r.route_id = t.route_id
    JOIN read_csv_auto('data/google_bus/calendar.txt') c
        ON t.service_id = c.service_id
    WHERE r.route_type = 3
    AND c.monday = 1
    AND c.tuesday = 1
    AND c.wednesday = 1
    GROUP BY r.route_short_name, r.route_long_name
    ORDER BY weekday_trips DESC
    LIMIT 20
""").df()

,route_short_name,route_long_name,weekday_trips
0,23,11th Market to Chestnut Hill,225
1,52,49th-Woodlnd to 54th-City/Parksd Lp,223
2,47,Whitman Plaza to 5th-Godfrey,221
3,42,2nd-Spruce to 61st-Pine/Wycombe,216
4,63,Lanknau/Overbrk to ColCom/FdCtr,209
5,21,Columbus-Dock to 69th St TC,202
6,45,Broad-Oregon to 12th-Vine,199
7,82,Wisshkn TC/Henry-Midvl to FrankfdTC,198
8,18,Fox Chase to Cedarbrook Plaza,195
9,60,35th-Allegheny to Rich-Westmoreland,195


In [8]:
con.sql("""
    SELECT
        st.arrival_time,
        COUNT(*) as trips_at_this_time
    FROM read_csv_auto('data/google_bus/stop_times.txt') st
    JOIN read_csv_auto('data/google_bus/trips.txt') t
        ON st.trip_id = t.trip_id
    JOIN read_csv_auto('data/google_bus/routes.txt') r
        ON t.route_id = r.route_id
    JOIN read_csv_auto('data/google_bus/calendar.txt') c
        ON t.service_id = c.service_id
    WHERE r.route_short_name = '23'
    AND st.stop_sequence = 1
    AND c.monday = 1
    AND c.tuesday = 1
    AND c.wednesday = 1
    GROUP BY st.arrival_time
    ORDER BY st.arrival_time
""").df()

,arrival_time,trips_at_this_time
0,03:20:00,1
1,04:00:00,1
2,04:10:00,1
3,04:23:00,1
4,04:40:00,2
...,...,...
214,26:09:00,1
215,26:16:00,1
216,26:39:00,1
217,26:52:00,1


In [9]:
con.sql("""
    WITH trip_departures AS (
        SELECT
            st.arrival_time,
            t.trip_id
        FROM read_csv_auto('data/google_bus/stop_times.txt') st
        JOIN read_csv_auto('data/google_bus/trips.txt') t
            ON st.trip_id = t.trip_id
        JOIN read_csv_auto('data/google_bus/routes.txt') r
            ON t.route_id = r.route_id
        JOIN read_csv_auto('data/google_bus/calendar.txt') c
            ON t.service_id = c.service_id
        WHERE r.route_short_name = '23'
        AND st.stop_sequence = 1
        AND c.monday = 1
        AND c.tuesday = 1
        AND c.wednesday = 1
    ),
    gtfs_times AS (
        SELECT
            trip_id,
            arrival_time,
            CAST(
                SPLIT_PART(arrival_time, ':', 1) AS INTEGER) * 3600 +
                CAST(SPLIT_PART(arrival_time, ':', 2) AS INTEGER) * 60 +
                CAST(SPLIT_PART(arrival_time, ':', 3) AS INTEGER
            ) AS seconds_since_midnight
        FROM trip_departures
    )
    SELECT
        trip_id,
        arrival_time,
        seconds_since_midnight,
        LAG(seconds_since_midnight) OVER (ORDER BY seconds_since_midnight) AS prev_departure_seconds,
        ROUND(
            (seconds_since_midnight - LAG(seconds_since_midnight) OVER (ORDER BY seconds_since_midnight)) / 60.0
        , 1) AS headway_minutes
    FROM gtfs_times
    ORDER BY seconds_since_midnight
""").df()

,trip_id,arrival_time,seconds_since_midnight,prev_departure_seconds,headway_minutes
0,662156,03:20:00,12000,<NA>,NaN
1,662157,04:00:00,14400,12000,40.0
2,662098,04:10:00,15000,14400,10.0
3,662090,04:23:00,15780,15000,13.0
4,662099,04:40:00,16800,15780,17.0
...,...,...,...,...,...
220,662154,26:09:00,94140,92580,26.0
221,662039,26:16:00,94560,94140,7.0
222,662222,26:39:00,95940,94560,23.0
223,662040,26:52:00,96720,95940,13.0


In [10]:
con.sql("""
    WITH trip_departures AS (
        SELECT
            st.arrival_time,
            t.trip_id
        FROM read_csv_auto('data/google_bus/stop_times.txt') st
        JOIN read_csv_auto('data/google_bus/trips.txt') t
            ON st.trip_id = t.trip_id
        JOIN read_csv_auto('data/google_bus/routes.txt') r
            ON t.route_id = r.route_id
        JOIN read_csv_auto('data/google_bus/calendar.txt') c
            ON t.service_id = c.service_id
        WHERE r.route_short_name = '23'
        AND st.stop_sequence = 1
        AND c.monday = 1
        AND c.tuesday = 1
        AND c.wednesday = 1
    ),
    gtfs_times AS (
        SELECT
            trip_id,
            arrival_time,
            CAST(SPLIT_PART(arrival_time, ':', 1) AS INTEGER) * 3600 +
            CAST(SPLIT_PART(arrival_time, ':', 2) AS INTEGER) * 60 +
            CAST(SPLIT_PART(arrival_time, ':', 3) AS INTEGER) AS seconds_since_midnight
        FROM trip_departures
    )
    SELECT
        trip_id,
        arrival_time,
        seconds_since_midnight,
        ROUND(
            (seconds_since_midnight - LAG(seconds_since_midnight) 
            OVER (ORDER BY seconds_since_midnight)) / 60.0
        , 1) AS headway_minutes
    FROM gtfs_times
    WHERE seconds_since_midnight BETWEEN 25200 AND 68400
    ORDER BY seconds_since_midnight
""").df()

,trip_id,arrival_time,seconds_since_midnight,headway_minutes
0,662110,07:02:00,25320,NaN
1,662111,07:08:00,25680,6.0
2,662152,07:09:00,25740,1.0
3,662112,07:14:00,26040,5.0
4,662197,07:19:00,26340,5.0
...,...,...,...,...
144,662204,18:35:00,66900,8.0
145,662062,18:42:00,67320,7.0
146,662206,18:46:00,67560,4.0
147,662063,18:58:00,68280,12.0


In [11]:
con.sql("""
    WITH trip_departures AS (
        SELECT
            st.arrival_time,
            t.trip_id
        FROM read_csv_auto('data/google_bus/stop_times.txt') st
        JOIN read_csv_auto('data/google_bus/trips.txt') t
            ON st.trip_id = t.trip_id
        JOIN read_csv_auto('data/google_bus/routes.txt') r
            ON t.route_id = r.route_id
        JOIN read_csv_auto('data/google_bus/calendar.txt') c
            ON t.service_id = c.service_id
        WHERE r.route_short_name = '47'
        AND st.stop_sequence = 1
        AND c.monday = 1
        AND c.tuesday = 1
        AND c.wednesday = 1
    ),
    gtfs_times AS (
        SELECT
            trip_id,
            arrival_time,
            CAST(SPLIT_PART(arrival_time, ':', 1) AS INTEGER) * 3600 +
            CAST(SPLIT_PART(arrival_time, ':', 2) AS INTEGER) * 60 +
            CAST(SPLIT_PART(arrival_time, ':', 3) AS INTEGER) AS seconds_since_midnight
        FROM trip_departures
    )
    SELECT
        trip_id,
        arrival_time,
        seconds_since_midnight,
        ROUND(
            (seconds_since_midnight - LAG(seconds_since_midnight)
            OVER (ORDER BY seconds_since_midnight)) / 60.0
        , 1) AS headway_minutes
    FROM gtfs_times
    WHERE seconds_since_midnight BETWEEN 25200 AND 68400
    ORDER BY seconds_since_midnight
""").df()

,trip_id,arrival_time,seconds_since_midnight,headway_minutes
0,728174,07:00:00,25200,NaN
1,728279,07:02:00,25320,2.0
2,728280,07:09:00,25740,7.0
3,728175,07:10:00,25800,1.0
4,728281,07:17:00,26220,7.0
...,...,...,...,...
151,728261,18:33:00,66780,6.0
152,728157,18:40:00,67200,7.0
153,728342,18:43:00,67380,3.0
154,728151,18:54:00,68040,11.0


## Scope Decision

Routes: 23, 47
Mode: Bus only
Service: Weekday service (Monday-Wednesday service patterns)
Time window: 7:00am - 7:00pm (peak and interpeak)
KPI: On-Time Performance (OTP)
Robustness dimensions:
  - Lateness threshold (1 min, 3 min, 5 min)
  - Stop selection (timepoints only vs all stops)
  - Cancelled trip treatment (exclude vs count as late)